In [1]:
import math
from typing import Callable
from scipy.stats import chi2
import numpy as np

def est_T(frecuencias: list[int], probabilidades: list[float], n: int, K: int):
    t = 0
    for k in range(K):
        t = t + (frecuencias[k] - n * probabilidades[k]
                 )**2 / (n * probabilidades[k])

    return t

def gen_frecuencias_binomial(probabilidades: list[float], n: int, K: int) -> list[int]:
    n_res = n
    freq = []
    p = probabilidades[0]
    for i in range(K - 1):
        N_i = np.random.binomial(n_res, p)
        freq.append(N_i)

        n_res -= N_i
        p = probabilidades[i + 1] / (1 - sum(probabilidades[:(i + 1)]))

    freq.append(n_res)

    return freq

### Ejercicio 2.
Para verificar que cierto dado no estaba trucado, se registraron $1000$ lanzamientos, resultando que el número de veces que el dado arrojó el valor $i$ ($i = 1, 2, 3, 4, 5, 6$) fue, respectivamente, $158, 172, 164, 181, 160, 165$. Aproximar el *p−valor* de la prueba: “el dado es honesto”

a) utilizando la prueba de Pearson con aproximación chi-cuadrada,

b) realizando una simulación.

---

In [2]:
frecuencias = [158, 172, 164, 181, 160, 165]
probabilidades = [1/6 for _ in range(6)]

assert sum(frecuencias) == 1_000

### Análisis
Si se asume que el dado es honesto, esto hace que $X \in \{1, 2, 3, 4, 5, 6\}$, con $X \sim U(\{1, 2, \cdots, 6\})$ y que $P(X = x) = 1/6$.

Entonces tenemos que $F = U(\{1, 2, \cdots, 6\})$ y que $ H_0 = \text{el dado es honesto} $


In [3]:
# Punto a
def punto_a(frecuencias: list[int], probabilidades: list[float]):
    K = len(frecuencias)
    n = sum(frecuencias)
    
    t_obs = est_T(frecuencias, probabilidades, n, K)
    p_valor = chi2.sf(t_obs, df=(K - 1))

    # print("p-valor: "", 1 - chi2.cdf(t_obs, df=(K - 1)))
    print("p-valor: ", p_valor)


punto_a(frecuencias, probabilidades)

p-valor:  0.8237195392577814


In [4]:
# Punto b
def punto_b(
    frecuencias: list[int],
    probabilidades: list[float],
    gen_freq: Callable[[list[float]], list[float]],
    N_SIM: int = 10_000
):

    n = sum(frecuencias)
    K = len(frecuencias)
    t_obs = est_T(frecuencias, probabilidades, n, K)

    p_valor = 0
    for _ in range(N_SIM):
        # generar lista de frecuencias utilizando la lista de probabilidades
        freq = gen_freq(probabilidades, n, K)

        # calculo el estadístico para la muestra generada
        t = est_T(freq, probabilidades, n, K)

        if t_obs <= t:
            p_valor += 1

    p_valor = p_valor / N_SIM
    print("p-valor: ", p_valor)

# Utilizo gen_frecuencias_binomial ya que al no estimar ningún parámetro, no
# necesito generar la muestra.
punto_b(frecuencias, probabilidades, gen_frecuencias_binomial)

p-valor:  0.8281


Como *p-valor* no es bajo, no se tiene evidencia suficiente para rechazar $H_0$. Como era alto, podría pensarse que esta ocurriendo algo raro, pero por simulación, sabemos que esta bien.